In [1]:
import glob

csv_files = glob.glob("/kaggle/input/**/*.csv", recursive=True)

print("CSV files found:")
print(csv_files)

file_path = csv_files[0]

df = pd.read_csv(file_path, low_memory=False)
# Remove accidental spaces from column names
df.columns = df.co/kaggle/input/datasets/sweety18/cicids2017-full-dataset/combine.csvlumns.str.strip()

print("Dataset loaded successfully")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst five rows:")
print(df.head())

CSV files found:
['/kaggle/input/datasets/sweety18/cicids2017-full-dataset/combine.csv']


NameError: name 'pd' is not defined

In [5]:
print("=== DATASET ANALYSIS ===")

print("\nRows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nNumber of classes:", df["Label"].nunique())

print("\nClass distribution:")
print(df["Label"].value_counts())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

print("\nColumns containing missing values:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

=== DATASET ANALYSIS ===

Rows: 2214469
Columns: 79

Number of classes: 10

Class distribution:
Label
BENIGN              1672837
DoS Hulk             231073
PortScan             158930
DDoS                 128027
DoS GoldenEye         10293
DoS slowloris          5796
DoS Slowhttptest       5499
Bot                    1966
Infiltration             36
Heartbleed               11
Name: count, dtype: int64

Duplicate rows: 271598

Total missing values: 1215

Columns containing missing values:
Flow Duration                  1
Total Fwd Packets              1
Total Backward Packets         1
Total Length of Fwd Packets    1
Total Length of Bwd Packets    1
                              ..
Idle Mean                      1
Idle Std                       1
Idle Max                       1
Idle Min                       1
Label                          1
Length: 78, dtype: int64


In [6]:
duplicate_count = df.duplicated().sum()
missing_by_column = df.isnull().sum()

print("Duplicate rows:", duplicate_count)
print("Total missing values:", missing_by_column.sum())

print("\nColumns with missing values:")
print(missing_by_column[missing_by_column > 0])

Duplicate rows: 271598
Total missing values: 1215

Columns with missing values:
Flow Duration                  1
Total Fwd Packets              1
Total Backward Packets         1
Total Length of Fwd Packets    1
Total Length of Bwd Packets    1
                              ..
Idle Mean                      1
Idle Std                       1
Idle Max                       1
Idle Min                       1
Label                          1
Length: 78, dtype: int64


In [7]:
# Create a separate cleaned copy
df_clean = df.copy()

# Convert infinite values into missing values
df_clean = df_clean.replace([np.inf, -np.inf], np.nan)

# Remove duplicate network records
df_clean = df_clean.drop_duplicates()

# Remove rows that have no class label
df_clean = df_clean.dropna(subset=["Label"])

# Replace remaining missing numeric values with 0
numeric_columns = df_clean.select_dtypes(include=np.number).columns
df_clean[numeric_columns] = df_clean[numeric_columns].fillna(0)

# Final cleaning check
print("Original shape:", df.shape)
print("Cleaned shape:", df_clean.shape)
print("Duplicate rows remaining:", df_clean.duplicated().sum())
print("Missing values remaining:", df_clean.isnull().sum().sum())

Original shape: (2214469, 79)
Cleaned shape: (1942870, 79)
Duplicate rows remaining: 4
Missing values remaining: 0


In [8]:
df_clean = df_clean.drop_duplicates()

print("Final cleaned shape:", df_clean.shape)
print("Duplicate rows remaining:", df_clean.duplicated().sum())
print("Missing values remaining:", df_clean.isnull().sum().sum())

Final cleaned shape: (1942866, 79)
Duplicate rows remaining: 0
Missing values remaining: 0


In [9]:
from sklearn.model_selection import train_test_split

# Keep the original cleaned data unchanged
df_model = df_clean.copy()

# Create target: 0 = BENIGN traffic, 1 = any attack traffic
df_model["Binary_Label"] = np.where(
    df_model["Label"].str.strip().str.upper() == "BENIGN",
    0,
    1
)

# Take a manageable sample for the first machine-learning model
df_sample = df_model.sample(n=300000, random_state=42)

# X = network traffic details; y = normal/attack answer
X = df_sample.drop(columns=["Label", "Binary_Label"])
y = df_sample["Binary_Label"]

# Split: 80% for learning, 20% for final testing
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Feature columns:", X.shape[1])
print("Training records:", X_train.shape[0])
print("Testing records:", X_test.shape[0])

print("\nBinary classes:")
print(y.value_counts().rename({0: "BENIGN", 1: "ATTACK"}))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

Feature columns: 78
Training records: 240000
Testing records: 60000

Binary classes:
Binary_Label
BENIGN    236239
ATTACK     63761
Name: count, dtype: int64

Training class distribution:
Binary_Label
0    0.787462
1    0.212537
Name: proportion, dtype: float64


In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Create and train the model
model = RandomForestClassifier(
    n_estimators=100,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# Predict labels for unseen test traffic
y_pred = model.predict(X_test)

# Important cybersecurity metrics: attack class = 1
print("=== ATTACK DETECTION METRICS ===")
print("Precision:", precision_score(y_test, y_pred, pos_label=1))
print("Recall:", recall_score(y_test, y_pred, pos_label=1))
print("F1-score:", f1_score(y_test, y_pred, pos_label=1))

print("\n=== MACRO METRICS ===")
print("Macro Precision:", precision_score(y_test, y_pred, average="macro"))
print("Macro Recall:", recall_score(y_test, y_pred, average="macro"))
print("Macro F1-score:", f1_score(y_test, y_pred, average="macro"))

print("\n=== CLASS-WISE REPORT ===")
print(classification_report(
    y_test,
    y_pred,
    target_names=["BENIGN", "ATTACK"]
))

print("\n=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

=== ATTACK DETECTION METRICS ===
Precision: 0.9964630983258665
Recall: 0.9941969887076537
F1-score: 0.9953287536800786

=== MACRO METRICS ===
Macro Precision: 0.9974489275921906
Macro Recall: 0.9966222837205725
Macro F1-score: 0.9970349137350406

=== CLASS-WISE REPORT ===
              precision    recall  f1-score   support

      BENIGN       1.00      1.00      1.00     47248
      ATTACK       1.00      0.99      1.00     12752

    accuracy                           1.00     60000
   macro avg       1.00      1.00      1.00     60000
weighted avg       1.00      1.00      1.00     60000


=== CONFUSION MATRIX ===
[[47203    45]
 [   74 12678]]
